In [2]:

%pip install plotly pandas nbformat


'pip' is not recognized as an internal or external command,
operable program or batch file.


In [3]:


import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("student_management_dataset.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df.info())


Dataset shape: (1000, 23)
   Student_ID  Age  Gender Ethnicity Socioeconomic_Status  \
0           1   24  Female  Hispanic               Medium   
1           2   21  Female     Asian                  Low   
2           3   28    Male     Black               Medium   
3           4   25  Female     Black               Medium   
4           5   22    Male     Other               Medium   

  Parental_Education_Level  Disability_Status   GPA Past_Academic_Performance  \
0                 Master’s                  0  3.26             Above Average   
1              High School                  0  2.04             Below Average   
2               Bachelor’s                  0  2.42                   Average   
3               Bachelor’s                  0  2.48                   Average   
4               Bachelor’s                  0  3.94             Below Average   

   Current_Semester_Performance  ...  Attendance_Rate  \
0                            90  ...        62.150096   
1     

In [4]:

# Check missing values
print("Missing values:\n", df.isnull().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Fill missing numerical values with their median
for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown")

print("Dataset cleaned successfully!")
print("Rows and columns:", df.shape)


Missing values:
 Student_ID                      0
Age                             0
Gender                          0
Ethnicity                       0
Socioeconomic_Status            0
Parental_Education_Level        0
Disability_Status               0
GPA                             0
Past_Academic_Performance       0
Current_Semester_Performance    0
Courses_Failed                  0
Credits_Completed               0
Study_Hours_per_Week            0
Attendance_Rate                 0
Number_of_Late_Submissions      0
Class_Participation_Score       0
Online_Learning_Hours           0
Library_Usage_Hours             0
Disciplinary_Actions            0
Social_Engagement_Score         0
Mental_Health_Score             0
Extracurricular_Activities      0
Dropout_Risk                    0
dtype: int64
Dataset cleaned successfully!
Rows and columns: (1000, 23)


C:\Users\spoorthi\AppData\Local\Temp\ipykernel_11092\3834718609.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [5]:

fig1 = px.histogram(
    df,
    x="GPA",
    nbins=20,
    title="Distribution of Student GPA",
    template="plotly_white"
)

fig1.update_layout(
    xaxis_rangeslider_visible=True,
    xaxis_title="GPA",
    yaxis_title="Number of Students"
)

fig1.show()


In [6]:

fig2 = px.scatter(
    df,
    x="GPA",
    y="Attendance_Rate",
    color="Dropout_Risk",
    hover_data=[
        "Student_ID",
        "Gender",
        "Study_Hours_per_Week",
        "Courses_Failed"
    ],
    title="GPA vs Attendance Rate by Dropout Risk",
    template="plotly_white"
)

fig2.show()


In [7]:

fig3 = px.box(
    df,
    x="Dropout_Risk",
    y="GPA",
    color="Dropout_Risk",
    points="all",
    title="GPA Distribution by Dropout Risk",
    template="plotly_white"
)

fig3.show()


In [8]:

fig4 = px.scatter_3d(
    df,
    x="GPA",
    y="Attendance_Rate",
    z="Study_Hours_per_Week",
    color="Dropout_Risk",
    hover_data=["Student_ID", "Courses_Failed"],
    title="3D Analysis of Student Performance",
    template="plotly_white"
)

fig4.show()



In [9]:

animation_df = df.dropna(
    subset=[
        "Age",
        "GPA",
        "Attendance_Rate",
        "Study_Hours_per_Week"
    ]
).copy()

animation_df["Age"] = animation_df["Age"].astype(int)

fig5 = px.scatter(
    animation_df,
    x="GPA",
    y="Attendance_Rate",
    animation_frame="Age",
    size="Study_Hours_per_Week",
    color="Dropout_Risk",
    hover_data=["Student_ID", "Courses_Failed"],
    range_x=[0, 4.5],
    range_y=[0, 100],
    size_max=25,
    title="Animated Student Performance by Age",
    template="plotly_white"
)

fig5.show()

# Animation compares student patterns across ages.
# A static chart shows all students at once.


In [10]:

fig6 = go.Figure()

fig6.add_trace(
    go.Scatter(
        x=df["GPA"],
        y=df["Attendance_Rate"],
        mode="markers",
        marker=dict(
            size=8,
            color=df["Dropout_Risk"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Dropout Risk")
        ),
        text=df["Student_ID"],
        hovertemplate=(
            "Student ID: %{text}<br>"
            "GPA: %{x}<br>"
            "Value: %{y}<extra></extra>"
        )
    )
)

fig6.update_layout(
    title="Student GPA vs Selected Metric",
    xaxis_title="GPA",
    yaxis_title="Attendance Rate (%)",
    template="plotly_white",
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Attendance Rate",
                    method="update",
                    args=[
                        {"y": [df["Attendance_Rate"]]},
                        {"yaxis.title": "Attendance Rate (%)"}
                    ]
                ),
                dict(
                    label="Study Hours",
                    method="update",
                    args=[
                        {"y": [df["Study_Hours_per_Week"]]},
                        {"yaxis.title": "Study Hours per Week"}
                    ]
                ),
                dict(
                    label="Courses Failed",
                    method="update",
                    args=[
                        {"y": [df["Courses_Failed"]]},
                        {"yaxis.title": "Courses Failed"}
                    ]
                )
            ],
            direction="down",
            x=1.02,
            y=1
        )
    ]
)

fig6.show()


In [11]:

from plotly.offline import plot

figures = [fig1, fig2, fig3, fig4, fig5, fig6]

with open(
    "Student_Management_Interactive_Analysis.html",
    "w",
    encoding="utf-8"
) as file:

    file.write("""
    <html>
    <head>
        <title>Student Management Analysis</title>
    </head>
    <body>
        <h1>Interactive Student Performance Dashboard</h1>
    """)

    for i, fig in enumerate(figures):
        file.write(
            plot(
                fig,
                output_type="div",
                include_plotlyjs=(i == 0)
            )
        )

    file.write("</body></html>")

print("HTML dashboard created successfully!")


HTML dashboard created successfully!
